In [2]:
# Find customer AS type based on Stanford ASdb
import pandas as pd
asn = "13335"
year = "2024"

# Load AS numbers from the text file
with open("/home/shyam/jupy/ddos_scrubber/data/final_confirmed_customer_ases_"+asn+"_"+year+".txt", "r") as file:
    as_list = [line.strip() for line in file.readlines()]

# Load ASdb CSV file into a pandas DataFrame
asdb = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/"+year+"-01_categorized_ases.csv", low_memory=False)

# Ensure AS numbers in the CSV are formatted correctly as strings and strip extra spaces
asdb['ASN'] = asdb['ASN'].astype(str).str.strip()

# # Put "AS" prefix prepend as the file contains only numbers
as_list = [f'AS{asn}' for asn in as_list]

# Match ASNs from the text file with the data in the CSV file
matched_asdb = asdb[asdb['ASN'].isin(as_list)]

# Convert dataframe object to list for finding ASes whose categories are not found in asdb dataset
matched_asdb_list = matched_asdb["ASN"].values.tolist()
unmatched_asdb = list(set(as_list) - set(matched_asdb_list))


# Save the results to a new file (optional)
output_file = "/home/shyam/jupy/ddos_scrubber/data/final_confirmed_customer_ases_categories_"+asn+"_"+year+".csv"
matched_asdb.to_csv(output_file, index=False)
print("Total %s number of matched ASNs and their categories are saved in a file %s " % (len(matched_asdb_list), output_file))
                   
print("%s ASNs not found in ASdb which are %s" %(len(unmatched_asdb), unmatched_asdb))


Total 621 number of matched ASNs and their categories are saved in a file /home/shyam/jupy/ddos_scrubber/data/final_confirmed_customer_ases_categories_13335_2024.csv 
11 ASNs not found in ASdb which are ['AS58789', 'AS38638', 'AS209555', 'AS37893', 'AS273584', 'AS58654', 'AS328867', 'AS399358', 'AS149294', 'AS55383', 'AS149648']


In [5]:
# Find customer AS type based on Stanford ASdb
# After TMA

# Find number of ASes categories
import pandas as pd
year = "2021"
day = "01"

month_order = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 
               'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
for mon in month_order:
    df = pd.read_csv("../data/after_tma/customers_ases_scrubber_all_"+day+"_"+mon+"_"+year+".csv")
    as_list = df["as"]
    
    # Load ASdb CSV file into a pandas DataFrame
    asdb = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/"+year+"-05_categorized_ases.csv", low_memory=False)

    # Ensure AS numbers in the CSV are formatted correctly as strings and strip extra spaces
    asdb['ASN'] = asdb['ASN'].astype(str).str.strip()

    # # Put "AS" prefix prepend as the file contains only numbers
    as_list = [f'AS{asn}' for asn in as_list]

    # Match ASNs from the text file with the data in the CSV file
    matched_asdb = asdb[asdb['ASN'].isin(as_list)]

    # Convert dataframe object to list for finding ASes whose categories are not found in asdb dataset
    matched_asdb_list = matched_asdb["ASN"].values.tolist()
    unmatched_asdb = list(set(as_list) - set(matched_asdb_list))


    # Save the results to a new file (optional)
    output_file = "/home/shyam/jupy/ddos_scrubber/data/after_tma/final_confirmed_customer_ases_categories_"+day+"_"+mon+"_"+year+".csv"
    matched_asdb.to_csv(output_file, index=False)
    print("In %s, %s: Total %s number of matched ASNs, %s ASNs not found." % (mon, year,len(matched_asdb_list), len(unmatched_asdb)))


In jan, 2021: Total 756 number of matched ASNs, 45 ASNs not found.
In feb, 2021: Total 775 number of matched ASNs, 47 ASNs not found.
In mar, 2021: Total 772 number of matched ASNs, 48 ASNs not found.
In apr, 2021: Total 789 number of matched ASNs, 49 ASNs not found.
In may, 2021: Total 814 number of matched ASNs, 49 ASNs not found.
In jun, 2021: Total 843 number of matched ASNs, 51 ASNs not found.
In jul, 2021: Total 873 number of matched ASNs, 50 ASNs not found.
In aug, 2021: Total 914 number of matched ASNs, 55 ASNs not found.
In sep, 2021: Total 936 number of matched ASNs, 60 ASNs not found.
In oct, 2021: Total 979 number of matched ASNs, 63 ASNs not found.
In nov, 2021: Total 1029 number of matched ASNs, 68 ASNs not found.
In dec, 2021: Total 1045 number of matched ASNs, 71 ASNs not found.


In [2]:
# After TMA
# Let's assume "Category 1 - Layer 1" contains the primary categories, like "Finance and Insurance"
# We will group by "Category 1 - Layer 1" and count the number of ASes in each category
# Replace 'Category 1 - Layer 1' with the actual column name for the categories if necessary

import pandas as pd

year = "2024"
day = "01"

# month_order = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 
#                'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

month_order = ['dec']

for mon in month_order:
    try:
        file_path = f"../data/after_tma/final_confirmed_customer_ases_categories_{day}_{mon}_{year}.csv"
        matched_ases = pd.read_csv(file_path)

        # === Layer 1 counts ===
        category_column = 'Category 1 - Layer 1'
        layer1_counts = matched_ases.groupby(category_column)['ASN'].count().reset_index()
        layer1_counts.columns = ['Category', 'ASN Count']
        layer1_counts.insert(0, 'Layer', 'Layer 1')

        # === Filter for IT category ===
        it_condition = (matched_ases[category_column] == "Computer and Information Technology")
        matched_ases_it = matched_ases[it_condition]

        # === Layer 2 counts within IT for Cloud & ISP only ===
        sub_category_column = 'Category 1 - Layer 2'
        target_subcategories = ["Internet Service Provider (ISP)", "Hosting and Cloud Provider"]

        it_filtered = matched_ases_it[matched_ases_it[sub_category_column].isin(target_subcategories)]
        it_sub_counts = it_filtered.groupby(sub_category_column)['ASN'].count().reset_index()
        it_sub_counts.columns = ['Category', 'ASN Count']
        it_sub_counts.insert(0, 'Layer', 'Layer 2 (IT only)')

        # === Compute "Other / Unknown" category for IT ===
        it_total = matched_ases_it['ASN'].count()
        known_sub_total = it_sub_counts['ASN Count'].sum()
        residual = it_total - known_sub_total

        residual_row = pd.DataFrame([{
            'Layer': 'Layer 2 (IT only)',
            'Category': 'IT after cloud_isp',
            'ASN Count': residual
        }])

        it_final_counts = pd.concat([it_sub_counts, residual_row], ignore_index=True)

        # === Combine Layer 1 and filtered Layer 2 counts ===
        combined_counts = pd.concat([layer1_counts, it_final_counts], ignore_index=True)

        # === Save to CSV ===
        output_path = f"../data/after_tma/final_confirmed_customer_ases_categories_counts_{day}_{mon}_{year}.csv"
#         combined_counts.to_csv(output_path, index=False)

        print(f"Saved combined counts for {mon} {year} to {output_path}")

    except FileNotFoundError:
        print(f"File not found for {mon} {year}, skipping.")
    except Exception as e:
        print(f"Error processing {mon} {year}: {e}")


Saved combined counts for dec 2024 to ../data/after_tma/final_confirmed_customer_ases_categories_counts_01_dec_2024.csv


In [18]:
it_filtered = matched_ases_it[matched_ases_it[sub_category_column].isin(target_subcategories)]
isps = it_filtered[it_filtered["Category 1 - Layer 2"] == "Internet Service Provider (ISP)"]["ASN"]
list(isps)

['AS8971',
 'AS6752',
 'AS15892',
 'AS20521',
 'AS5539',
 'AS15600',
 'AS21054',
 'AS5430',
 'AS12357',
 'AS20775',
 'AS15830',
 'AS25418',
 'AS29293',
 'AS9022',
 'AS34184',
 'AS34358',
 'AS34397',
 'AS34960',
 'AS34967',
 'AS35733',
 'AS6908',
 'AS41741',
 'AS42612',
 'AS42875',
 'AS43009',
 'AS43394',
 'AS43566',
 'AS43719',
 'AS44586',
 'AS47215',
 'AS48846',
 'AS50743',
 'AS51265',
 'AS51409',
 'AS51561',
 'AS51747',
 'AS51840',
 'AS56460',
 'AS197991',
 'AS198247',
 'AS198301',
 'AS198381',
 'AS198394',
 'AS198504',
 'AS199188',
 'AS199379',
 'AS60969',
 'AS199272',
 'AS60229',
 'AS200005',
 'AS60068',
 'AS59907',
 'AS201188',
 'AS200946',
 'AS200422',
 'AS204257',
 'AS204206',
 'AS50152',
 'AS202623',
 'AS206936',
 'AS206693',
 'AS51486',
 'AS206283',
 'AS206067',
 'AS205721',
 'AS41839',
 'AS205892',
 'AS62053',
 'AS43355',
 'AS57276',
 'AS200361',
 'AS210243',
 'AS209995',
 'AS35051',
 'AS208552',
 'AS208428',
 'AS42433',
 'AS212885',
 'AS211526',
 'AS211054',
 'AS206587',
 'A

In [3]:
# Let's assume "Category 1 - Layer 1" contains the primary categories, like "Finance and Insurance"
# We will group by "Category 1 - Layer 1" and count the number of ASes in each category
# Replace 'Category 1 - Layer 1' with the actual column name for the categories if necessary
category_column = 'Category 1 - Layer 1'

matched_ases = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/final_confirmed_customer_ases_categories_"+asn+"_"+year+".csv")



# Group by category and count the number of AS numbers in each group
category_counts = matched_ases.groupby(category_column)['ASN'].count()

category_column = 'Category 1 - Layer 1'

matched_ases = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/after_tma/final_confirmed_customer_ases_categories_"+asn+"_"+year+".csv")

condition = (matched_ases[category_column] == "Computer and Information Technology") 

matched_ases_information_technology = matched_ases.loc[condition]
# matched_ases_information_technology

category_column = "Category 1 - Layer 2"
information_technology_category_counts = matched_ases_information_technology.groupby(category_column)['ASN'].count()


print(information_technology_category_counts)

# Print the results
print("Number of ASes per category:")
print(category_counts)

# Optionally, save the result to a new CSV
category_counts.to_csv("/home/shyam/jupy/ddos_scrubber/data/final_confirmed_customer_ases_categories_counts_"+asn+"_"+year+".csv", header=True)

Number of ASes per category:
Category 1 - Layer 1
Agriculture, Mining, and Refineries (Farming, Greenhouses, Mining, Forestry, and Animal Farming)      8
Community Groups and Nonprofits                                                                      11
Computer and Information Technology                                                                 292
Construction and Real Estate                                                                         11
Education and Research                                                                               15
Finance and Insurance                                                                                84
Freight, Shipment, and Postal Services                                                               11
Government and Public Administration                                                                 14
Health Care Services                                                                                 15
Manufacturing 

In [10]:
matched_ases

,ASN,Category 1 - Layer 1,Category 1 - Layer 2,Category 2 - Layer 1,Category 2 - Layer 2,Category 3 - Layer 1,Category 3 - Layer 2,Category 4 - Layer 1,Category 4 - Layer 2,Category 5 - Layer 1,...,Category 74 - Layer 1,Category 74 - Layer 2,Category 75 - Layer 1,Category 75 - Layer 2,Category 76 - Layer 1,Category 76 - Layer 2,Category 77 - Layer 1,Category 77 - Layer 2,Category 78 - Layer 1,Category 78 - Layer 2
0,AS3295,Finance and Insurance,"Banks, Credit Card Companies, Mortgage Providers",Finance and Insurance,Insurance Carriers and Agencies,Finance and Insurance,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AS3298,Finance and Insurance,"Banks, Credit Card Companies, Mortgage Providers",Finance and Insurance,Insurance Carriers and Agencies,Finance and Insurance,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AS3299,Finance and Insurance,"Banks, Credit Card Companies, Mortgage Providers",Finance and Insurance,Insurance Carriers and Agencies,Finance and Insurance,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AS5540,Utilities (Excluding Internet Service),"Electric Power Generation, Transmission, Distr...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AS8435,Finance and Insurance,"Banks, Credit Card Companies, Mortgage Providers",Finance and Insurance,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1668,AS394358,Finance and Insurance,"Banks, Credit Card Companies, Mortgage Providers",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1669,AS396223,Computer and Information Technology,Internet Service Provider (ISP),Service,"Buildings, Repair, Maintenance (Pest Control, ...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1670,AS399566,Computer and Information Technology,Internet Service Provider (ISP),NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1671,AS400797,Computer and Information Technology,Internet Service Provider (ISP),Computer and Information Technology,Software Development,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [65]:
# For category with "Computer and Information Technology", find their subcategory that is "Category 1 - Layer 2"
# We will group by "Category 1 - Layer 2" and count the number of ASes in each category
# Replace 'Category 1 - Layer 1' with the actual column name for the categories if necessary
category_column = 'Category 1 - Layer 1'

matched_ases = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/final_confirmed_customer_ases_categories_"+asn+"_"+year+".csv")

condition = (matched_ases[category_column] == "Computer and Information Technology") 

matched_ases_information_technology = matched_ases.loc[condition]
# matched_ases_information_technology

category_column = "Category 1 - Layer 2"
information_technology_category_counts = matched_ases_information_technology.groupby(category_column)['ASN'].count()

# Print the results
print("Number of ASes per Computer and Information Technology category")
print(information_technology_category_counts)

# Optionally, save the result to a new CSV
category_counts.to_csv("/home/shyam/jupy/ddos_scrubber/data/final_confirmed_customer_ases_it_category_counts_"+asn+"_"+year+".csv", header=True)

Number of ASes per Computer and Information Technology category
Category 1 - Layer 2
Computer and Network Security      1
Internet Service Provider (ISP)    6
Software Development               1
Name: ASN, dtype: int64
